# ML S4 · Notebook 04 — Practice: A Classification Pipeline, End to End

| | |
|---|---|
| **Session ID** | ML S4 · Notebook 04 of 04 |
| **Course position** | Machine Learning — Session 4 of 10 (Classification &amp; the Model Family Map) |
| **Block types** | ⚙️ **Delivery block** — AI assistance expected and encouraged. The assessed skill is specifying what you want, then verifying you actually got it. No new concepts appear in this notebook. |
| **Prerequisites** | **Notebooks 01–03** of this session · ML S3 (splitting, CV, confusion matrix, statistical comparison) |
| **Connects back to** | **ML S2** — baselines · **ML S3** — the entire evaluation discipline · **Notebook 02** — why a tree beats a line on this data · **Notebook 03** — tuning honestly |
| **Connects forward to** | **ML S5** (ensembles and gradient boosting — the model that should win this bake-off) · **ML S6** (thresholds and imbalance) · **ML S9** (deployment) · **ML S10** (CEP clinic) |
| **CEP linkage** | This is the Employee Turnover CEP in rehearsal. That brief asks for a stratified split, cross-validated training of three model families, a justified choice of metric, and a recommendation. Everything except SMOTE and the risk bands is exercised here. |
| **Run requirements** | `pandas`, `numpy`, `scikit-learn`, `matplotlib`. Run `ML_S4_00_dataset.ipynb` first. |
| **Checkpoint file** | `subscription_churn.csv` |


## Learning Objectives

By the end of this notebook you will be able to:

1. **Execute a complete classification workflow** from raw CSV to a defended recommendation, without being prompted step by step.
2. **Choose and justify an evaluation metric** before looking at any model output.
3. **Compare model families fairly** on identical folds with identical preprocessing.
4. **Tune a shortlist honestly** and report a number that selection did not touch.
5. **Defend a model choice on grounds other than the highest score** — which is the actual deliverable of both ML CEPs.

## Table of Contents

| § | Section | Type |
|---|---|---|
| 1 | The brief | Framing |
| 2 | Step 1 — Frame the decision before touching data | ⚙️ Delivery |
| 3 | Step 2 — Load, split, and set the baseline | ⚙️ Delivery |
| 4 | Step 3 — One preprocessing pipeline, reused by everything | ⚙️ Delivery |
| 5 | Step 4 — The bake-off across families | ⚙️ Delivery |
| 6 | Step 5 — Tune the shortlist | ⚙️ Delivery |
| 7 | Step 6 — The honest number | ⚙️ Delivery |
| 8 | Step 7 — The recommendation | ⚙️ Delivery |
| 9 | What you deliberately did not do today | Framing |
| — | Common Pitfalls · FAQ · **Session Conclusion** · Transition | Wrap-up |

# 1. The brief

> **From:** Head of Retention
> **To:** you
>
> We lose about a fifth of our subscriber base each year. I want to know which customers are at risk *before* they cancel, so my team can call them.
>
> My team can work through roughly **300 calls a month**. I do not want a list of 2,000 names.
>
> Tell me which model you would put into production and why. I will be asked to justify the spend, so "it scored highest" is not an answer I can use.

That is a realistic brief: a decision, a constraint, and a demand for justification. Notice what it does *not* contain — any mention of accuracy, AUC, or algorithms. Translating it into those terms is your job, and it is the first step.

**How to work this notebook.** Each step states a goal and gives you the code. Use AI assistance freely — that is the point of a delivery block. What you are practising is knowing what to ask for and recognising when the answer is wrong. Every step ends with a **verification** you should actually read rather than scroll past.

# 2. Step 1 — Frame the decision before touching data

Before any code, four questions. This takes three minutes and saves hours.

**What is the unit of analysis?** One customer, at one point in time. One row, one prediction.

**What is the target?** `churned` — did this customer cancel. Binary. Roughly 20% positive.

**What decision does the output drive?** Whether a retention agent calls this customer. That is a ranked list with a capacity limit, not a yes/no verdict on all 12,000 people.

**What does each error cost?**

| Error | What it means here | Cost |
|---|---|---|
| **False negative** | A customer who will churn is not flagged | Lost subscriber — the expensive one |
| **False positive** | A customer who would have stayed gets a call | One agent's time, maybe a small discount |

The asymmetry is stark and it decides your metric. **Missing a churner costs far more than an unnecessary call.** That argues for prioritising **recall** over precision.

But it does not argue for recall *alone*, and here is where the brief's constraint earns its place: a model that flags everyone achieves perfect recall and is useless, because the team can only make 300 calls. **What you actually need is a model that ranks well** — one where the customers most likely to churn appear at the top of the list.

That is precisely what **ROC AUC** and **average precision** measure: quality of ranking, independent of where you eventually draw the line. So:

> **Metric decision, made before seeing any results:** tune and compare on **ROC AUC**, because the deliverable is a ranked call list. Report recall and precision alongside it for interpretability. Do **not** use accuracy — the majority-class floor is about 80%, so accuracy would call a model that predicts "nobody churns" a B-grade success.

**Where exactly to draw the line at 300 calls is a threshold decision, and that is Session 6.** Today ends with a ranking, deliberately.

### ✅ Quick Check

Before running anything: your colleague suggests optimising for precision, arguing that wasted calls annoy customers. Give one sentence for and one against.

<details>
<summary>Answer</summary>

**For:** precision is the fraction of flagged customers who really would churn, so optimising it keeps the call list clean and respects agents' time — and a retention call to a happy customer can genuinely plant the idea of leaving.

**Against:** precision ignores everyone you missed. A model flagging 30 customers, all of whom churn, has perfect precision, uses 10% of the team's capacity, and lets hundreds of churners leave unnoticed. Given the cost asymmetry, that is the worse failure.

**The resolution:** neither, as the primary. Optimise the *ranking* with AUC, then choose an operating point that fills 300 calls — which is exactly the Session 6 conversation.

</details>


# 3. Step 2 — Load, split, and set the baseline

Two disciplines from earlier sessions, applied without discussion because they are settled:

- **Stratified split**, so the ~20% positive rate is preserved in both halves (S3).
- **A baseline before any model**, so "good" has a reference point (S2).

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score,
                             precision_score, confusion_matrix,
                             classification_report, average_precision_score)
from scipy.stats import loguniform

churn = pd.read_csv('subscription_churn.csv')
X = churn.drop(columns='churned')
y = churn['churned']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"rows          : {len(churn):,}")
print(f"train / test  : {len(X_tr):,} / {len(X_te):,}")
print(f"churn rate    : overall {y.mean():.3f} | train {y_tr.mean():.3f} | test {y_te.mean():.3f}")
print(f"  -> stratification held: the three rates agree to within a fraction of a percent")

rows          : 12,000
train / test  : 9,600 / 2,400
churn rate    : overall 0.199 | train 0.199 | test 0.198
  -> stratification held: the three rates agree to within a fraction of a percent


In [2]:
# The baseline. Two versions, because they answer different questions.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

majority = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
stratified = DummyClassifier(strategy='stratified', random_state=0).fit(X_tr, y_tr)

print(f"majority-class accuracy on test : {accuracy_score(y_te, majority.predict(X_te)):.4f}")
print(f"majority-class recall on test   : {recall_score(y_te, majority.predict(X_te), zero_division=0):.4f}")
print(f"random-ranking AUC on test      : {roc_auc_score(y_te, stratified.predict_proba(X_te)[:, 1]):.4f}")
print()
print("Read these as the floor. Any model must clear the AUC of 0.5 by a wide margin")
print("to be worth deploying, and the ~0.80 accuracy figure must never be quoted as success.")

majority-class accuracy on test : 0.8017
majority-class recall on test   : 0.0000
random-ranking AUC on test      : 0.5011

Read these as the floor. Any model must clear the AUC of 0.5 by a wide margin
to be worth deploying, and the ~0.80 accuracy figure must never be quoted as success.


**Verification to actually read.** The three churn rates should agree to within a fraction of a percentage point — that is stratification working. They will not be bit-identical, because 2,400 rows cannot split a 19.9% rate perfectly. The majority-class accuracy should sit near 0.80 with a recall of exactly 0.000: a model that flags nobody is 80% accurate and catches zero churners.

Keep that number visible. It is the single clearest argument for why the metric decision in Step 1 had to come first.


# 4. Step 3 — One preprocessing pipeline, reused by everything

The fairness requirement: **every model gets identical preprocessing on identical folds.** One `ColumnTransformer`, built once, dropped into every pipeline.

This also handles the leak S3 warned about. Because scaling lives *inside* the pipeline, it is refitted on each training fold rather than on the whole dataset. Nothing from a validation fold reaches the scaler.

In [3]:
cat_cols = ['contract_type', 'region']
num_cols = [c for c in X.columns if c not in cat_cols]

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
])

print(f"numeric ({len(num_cols)}) : {num_cols}")
print(f"categorical ({len(cat_cols)}) : {cat_cols}")

# Verify the output width is what you expect before trusting anything downstream.
prep.fit(X_tr)
print(f"\nfeature count after preprocessing: {prep.transform(X_tr).shape[1]}")
print("  10 numeric + (3 contract - 1 dropped) + (4 region - 1 dropped) = 10 + 2 + 3 = 15")

numeric (10) : ['tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets_6m', 'avg_monthly_usage_gb', 'late_payments_12m', 'has_premium_support', 'num_services', 'age', 'satisfaction_score']
categorical (2) : ['contract_type', 'region']

feature count after preprocessing: 15
  10 numeric + (3 contract - 1 dropped) + (4 region - 1 dropped) = 10 + 2 + 3 = 15


**Verification.** The arithmetic in that last line should match the printed number. If it does not, something is being encoded that you did not intend — a numeric column silently read as text, most commonly. This one-line check catches a whole class of quiet bugs, and it costs nothing.


# 5. Step 4 — The bake-off across families

Five families, default settings, same folds, same metric. **No tuning yet** — this is a screening pass to decide what deserves the tuning budget.

Defaults are a legitimate first pass. A family that is hopeless at defaults rarely becomes the winner after tuning, and screening at defaults is far cheaper than tuning everything.

In [4]:
candidates = {
    'Logistic regression': LogisticRegression(max_iter=3000),
    'Decision tree':       DecisionTreeClassifier(random_state=0),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB':         GaussianNB(),
}

rows = []
for name, est in candidates.items():
    pipe = Pipeline([('prep', prep), ('model', est)])
    t0 = time.time()
    scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    rows.append({
        'model': name,
        'CV AUC': round(scores.mean(), 4),
        'fold std': round(scores.std(), 4),
        'seconds': round(time.time() - t0, 1),
    })

bakeoff = pd.DataFrame(rows).sort_values('CV AUC', ascending=False)
print(bakeoff.to_string(index=False))

              model  CV AUC  fold std  seconds
Logistic regression  0.8528    0.0059      6.9
        Gaussian NB  0.8110    0.0025      0.3
          KNN (k=5)  0.7738    0.0114      1.3
      Decision tree  0.6867    0.0128      8.7


**How to read this table — and this is the habit worth building.**

Do not simply take the top row. Ask three questions:

1. **Is the gap between first and second larger than the fold standard deviation?** If not, they are tied and you should prefer the simpler, faster, more interpretable one.
2. **Does any model do surprisingly badly?** That is diagnostic information about the data, not just about the model. Notebook 03's assumptions tell you what a poor score means for each family.
3. **What did it cost?** A model that takes 40× longer for a hundredth of AUC is not a good trade in production.

**What the table actually says, read top to bottom.**

**Logistic regression wins**, and comfortably. That is not a disappointment — it is the standard result on tabular data with mostly-linear signal, and it is exactly why it remains the default baseline in industry. Notebook 02 §9 already established why: contract type, tenure and satisfaction are strong smooth effects, and a line represents those exactly.

**Gaussian NB comes second**, which is genuinely surprising given that it is fitting bell curves to one-hot columns and to count features — a visible violation of its assumption. The explanation is the one from Notebook 03 §3: **Naive Bayes ranks better than it calibrates.** Its probabilities are distorted, but AUC only cares about the *order* of those probabilities, and the ordering survives. Had we screened on log-loss instead, it would have placed far worse. A model's rank in a table is a function of the metric, and swapping the metric reshuffles the table.

**KNN comes third, below Naive Bayes** — worse than the "closeness works fine in 15 dimensions" reasoning would predict. Two things are working against it: `k=5` is a small neighbourhood on 9,600 rows, making each prediction noisy, and the one-hot columns contribute crude 0/1 distances that dilute the informative numeric ones. Notebook 03 §2 showed `k=25` doing better.

**The single decision tree comes last**, and by a wide margin. At default settings it has no depth limit, so it grows until its leaves are pure — the memorisation failure from Notebook 02 §8, reproduced exactly. This is the most misleading row in the table, and the exercise below is about why.


### 🟢 Try It Yourself

The tree's score above is being held down by one unset hyperparameter.

Add a second tree to the bake-off with `max_depth=6` and re-run. Predict the direction of the change before you look.

<details>
<summary>Solution</summary>

```python
candidates['Decision tree (depth 6)'] = DecisionTreeClassifier(max_depth=6, random_state=0)
```

The constrained tree should improve substantially over the unconstrained one. The default tree memorises the training folds and generalises poorly; capping depth forces it to keep only the splits that carry real signal.

This is the point Notebook 03 §7 made about regularisation-shaped hyperparameters: `max_depth` is not fine-tuning, it is the difference between a memoriser and a model. It is also a preview of why Session 5 works — a forest of constrained trees, averaged, keeps the interaction-finding while controlling exactly this failure.

</details>


# 6. Step 5 — Tune the shortlist

Two candidates go forward: **logistic regression**, because it won the screen, and **the decision tree**, because it came last for a reason we have already diagnosed and can fix.

That second choice deserves defending, since it appears to contradict the screening logic. The rule "screen at defaults, promote the winners" is sound, but it carries an exception: **promote anything whose default score you have a specific, named reason to distrust.** Here that reason is concrete rather than hopeful — the exercise above showed a depth cap moving the tree substantially, and Notebook 02 §8 measured the same effect as a 0.21 train-CV gap collapsing once depth was constrained. The default tree is not a weak model; it is an unregularised one, and `max_depth` is the regularisation.

Gaussian NB, by contrast, scored respectably and is *not* promoted — it has essentially nothing to tune, and Notebook 03 §3 explained that its distorted probabilities make it a poor fit for a system that will eventually need a calibrated cut-off. A good screen promotes on diagnosis, not on rank order.

Random search, 25 draws each, same `cv`, same scoring — the fairness conditions from Notebook 03 §7.3.

In [5]:
searches = {}

searches['Logistic regression'] = RandomizedSearchCV(
    Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=4000))]),
    {'model__C': loguniform(1e-3, 1e2)},
    n_iter=25, cv=cv, scoring='roc_auc', random_state=0, n_jobs=-1)

searches['Decision tree'] = RandomizedSearchCV(
    Pipeline([('prep', prep), ('model', DecisionTreeClassifier(random_state=0))]),
    {'model__max_depth': np.arange(2, 21),
     'model__min_samples_leaf': np.arange(1, 121),
     'model__criterion': ['gini', 'entropy']},
    n_iter=25, cv=cv, scoring='roc_auc', random_state=0, n_jobs=-1)

tuned = []
for name, s in searches.items():
    s.fit(X_tr, y_tr)
    tuned.append({
        'model': name,
        'tuned CV AUC': round(s.best_score_, 4),
        'fold std': round(s.cv_results_['std_test_score'][s.best_index_], 4),
        'best params': {k.replace('model__', ''): v for k, v in s.best_params_.items()},
    })

print(pd.DataFrame(tuned).to_string(index=False))
print()
print("Compare each row against its default-settings CV AUC in the bake-off table above.")

              model  tuned CV AUC  fold std                                                      best params
Logistic regression        0.8529    0.0060                                        {'C': 42.460313017682125}
      Decision tree        0.8498    0.0104 {'min_samples_leaf': 18, 'max_depth': 7, 'criterion': 'entropy'}

Compare each row against its default-settings CV AUC in the bake-off table above.


**Verification.** Compare the tuned tree against the default tree from Step 4. That gap should be large — it is the `max_depth` effect. Then compare the tuned logistic regression against its default. That gap will be small, because `LogisticRegression` already applies L2 regularisation with a sensible `C` by default.

**Two models improving by very different amounts from tuning is the normal case, not an anomaly.** How much headroom tuning has depends entirely on how bad the defaults were for that family.


# 7. Step 6 — The honest number

Now, and only now, the test set. Everything up to this point used cross-validation inside the training data.

Recall Notebook 03 §8: `best_score_` is a maximum over 25 noisy attempts, and the number that goes in a report comes from data that took no part in selection.

The final column is the difference between the two. Notebook 03 §8.2 predicted what you will see here — on 9,600 rows with a 25-configuration search, **the difference comes out slightly negative**, because the refit-on-more-data effect outweighs the small selection effect at this data size. That is the expected result in this regime, not a sign that the discipline is unnecessary. §8.3 showed the same procedure producing a large positive gap on 400 rows.

In [6]:
final = []
for name, s in searches.items():
    est = s.best_estimator_
    proba = est.predict_proba(X_te)[:, 1]
    pred = est.predict(X_te)          # default 0.5 cut-off -- see the note below
    final.append({
        'model': name,
        'test AUC': round(roc_auc_score(y_te, proba), 4),
        'test AP': round(average_precision_score(y_te, proba), 4),
        'accuracy': round(accuracy_score(y_te, pred), 4),
        'recall': round(recall_score(y_te, pred), 4),
        'precision': round(precision_score(y_te, pred, zero_division=0), 4),
        'CV - test AUC': round(s.best_score_ - roc_auc_score(y_te, proba), 4),
    })

results = pd.DataFrame(final).sort_values('test AUC', ascending=False)
print(results.to_string(index=False))

              model  test AUC  test AP  accuracy  recall  precision  CV - test AUC
Logistic regression    0.8613   0.6630    0.8533  0.4727     0.6902        -0.0084
      Decision tree    0.8572   0.6294    0.8542  0.4790     0.6909        -0.0075


In [7]:
# Confusion matrices, at the DEFAULT 0.5 cut-off, for context.
for name, s in searches.items():
    pred = s.best_estimator_.predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    flagged = tp + fp
    print(f"--- {name} ---")
    print(f"  flagged {flagged} customers   caught {tp} of {tp + fn} churners   missed {fn}")
    print(f"  false alarms {fp}")
    print(f"  team capacity is 300 calls/month -> this list is "
          f"{'OVER' if flagged > 300 else 'within'} capacity")
    print()

--- Logistic regression ---
  flagged 326 customers   caught 225 of 476 churners   missed 251
  false alarms 101
  team capacity is 300 calls/month -> this list is OVER capacity

--- Decision tree ---
  flagged 330 customers   caught 228 of 476 churners   missed 248
  false alarms 102
  team capacity is 300 calls/month -> this list is OVER capacity



**The number that should stop you.** Look at how many customers each model flags at the default cut-off, against the 300-call capacity from the brief.

The default 0.5 cut-off was **not chosen by anyone**. It is the arbitrary midpoint that `predict()` uses because it has to use something. It bears no relationship to your team's capacity, to the cost asymmetry, or to the business problem.

**That dial is Session 6, and it is deliberately left untouched today.** Note it, do not turn it. Everything you have built here — a well-ranked probability from a fairly-tuned model — is exactly what Session 6 needs as its input. AUC was chosen in Step 1 precisely because it measures ranking quality *independent* of where that line eventually falls.


### 🖼️ Image Slot — Ranked call list against team capacity

**What to show:** A horizontal bar of 2,400 test customers sorted left-to-right by predicted churn probability, shaded dark for actual churners and light for stayers. A vertical dashed line at position 300 labelled 'team capacity'. Beneath it, two annotations: 'churners caught' spanning the dark bars left of the line, and 'churners missed' spanning dark bars to the right.

**Why here:** It converts an abstract AUC into the concrete deliverable the brief asked for, and it makes the Session 6 threshold question visible without answering it. Learners see immediately that the model's job is to push dark bars leftwards.

**Placement:** Directly after the capacity output, before the Step 7 recommendation section.

**Alt text:** Customers ranked by predicted churn risk, with a capacity cut-off at 300 showing which churners are caught and which are missed.

<!-- INSERT IMAGE: s4_04_ranked_call_list.png -->


# 8. Step 7 — The recommendation

The brief asked for a model and a justification that survives scrutiny. Here is the shape of a defensible answer.

**Template, to fill in from your own run:**

> I recommend **[model]** for the retention scoring system.
>
> **On performance:** it achieves a test AUC of **[x]**, against **0.5** for random ranking. The gap to the runner-up is **[y]**, which is [larger / smaller] than the fold-to-fold variation of **[std]** — so the two are [meaningfully different / statistically tied].
>
> **On the metric choice:** I optimised AUC rather than accuracy because the majority-class floor is **80%**, meaning a model that flags nobody would appear 80% accurate while catching zero churners. The deliverable is a ranked call list, and AUC measures ranking quality.
>
> **On the trade I accepted:** [e.g. I chose logistic regression despite the tree scoring marginally higher, because the difference is within noise and the retention team must justify individual calls. A coefficient table is auditable; a tree of depth 12 is not.]
>
> **On what remains:** the cut-off determining who actually gets called has not been set. That requires the team's capacity and the cost of a call versus the value of a retained subscriber. That is the next piece of work.

**What makes this defensible is not the score.** It is that the metric was justified before results were seen, the comparison accounted for noise, and the trade-off was named rather than hidden.

Both ML CEPs ask for exactly this. Employee Turnover explicitly requires you to justify the evaluation metric and to explain whether recall or precision matters more. A submission that reports three models and asserts the highest number wins is the most common resubmit.

### 🟢 Try It Yourself — the full rehearsal

Repeat this entire notebook with the target changed to `contract_type` (the three-class problem from Notebook 03 §6).

What has to change, and what does not?

<details>
<summary>Solution</summary>

**Unchanged:** the split (still stratified), the `ColumnTransformer`, the `Pipeline` structure, `cross_val_score`, `RandomizedSearchCV`, the tune-then-test discipline. Essentially all of the machinery.

**Must change:**

```python
y = churn['contract_type']
X = churn.drop(columns=['contract_type'])
cat_cols = ['region']                      # contract_type is now the target
```

- **Scoring.** `roc_auc` is binary by default. Use `roc_auc_ovr` for the multiclass version, or switch to `f1_macro`.
- **The metric report.** Use `classification_report` and read the per-class rows. Report macro, not micro — Notebook 03 §6.2 explains why micro would just be accuracy wearing a costume.
- **`predict_proba` output shape** becomes `(n, 3)` instead of `(n, 2)`, so `[:, 1]` no longer means anything sensible.
- **The business framing.** There is no "flag for a call" decision here. Without a decision attached, ask what the model is *for* before optimising anything — which is Step 1 all over again.

The transferable point: **the workflow is the stable asset.** Problem shape changes the metrics and the interpretation, and leaves the machinery almost entirely alone. That is why Notebook 01 taught the shapes as a map rather than as five separate methods.

</details>


# 9. What you deliberately did not do today

Naming the gaps matters as much as filling them, because each one is a scheduled session rather than an oversight.

| Not done | Why | Where it lands |
|---|---|---|
| Chose a decision threshold | Needs cost and capacity reasoning, not just a model | **ML S6** |
| Handled class imbalance (SMOTE, class weights) | An entire topic with its own leak trap | **ML S6** |
| Used an ensemble or gradient boosting | The model that should actually win this bake-off | **ML S5** |
| Compared models with a significance test | The machinery exists — you built it in S3 | **ML S5** bake-off |
| Engineered any features | The largest source of headroom, deliberately held back | **ML S5–S6** |
| Deployed anything | The pipeline object is already serialisable | **ML S9** |

**The honest summary of today's result:** you have a competently built, fairly evaluated, honestly reported model — and it is probably not the model you will ship. Session 5 introduces the family that usually wins on tabular data, and Session 6 turns the model into a decision.

What you have that most people skip is a **defensible process**. That does not change when the model does.


# Common Pitfalls

| Pitfall | What happens | The fix |
|---|---|---|
| Picking the metric after seeing results | You rationalise whichever number looks best | Decide in Step 1, in writing, before fitting |
| Reporting accuracy on imbalanced data | An 80%-accurate model that catches zero churners looks respectable | Report AUC plus recall; always quote the majority-class floor next to accuracy |
| Preprocessing outside the pipeline | Validation folds leak into the scaler | `ColumnTransformer` inside `Pipeline`, always |
| Tuning every candidate family | Wastes the budget on hopeless families | Screen at defaults, tune the top two |
| Reporting `best_score_` | Optimistic by construction | Report the test-set number |
| Reading `predict()` as the deliverable | Silently accepts an arbitrary 0.5 cut-off | Work with `predict_proba` and rank; set the cut-off deliberately in S6 |
| Declaring a winner inside the noise | The "best" model changes on re-run | Compare the gap against `fold std` before claiming a difference |

---

# FAQ

**My tree beat logistic regression. Should I ship the tree?**
Check the gap against the fold standard deviation first. If they are tied, prefer the simpler and more interpretable model. If the tree genuinely wins, that is the interaction and threshold effects being found — which Notebook 02 §9 predicted, and which Session 5 exploits properly.

**Should I use `class_weight='balanced'` here?**
It would likely help recall, and it is a legitimate tool — but it belongs with the rest of the imbalance toolkit in Session 6, where you can compare it against resampling and understand the trade. Adding it now without that context would be cargo-culting.

**Why not just use `RandomForestClassifier`?**
It should win. It is deliberately held for Session 5, where you will learn *why* it works rather than just observing that it does.

**How long should this take in a real job?**
The workflow above is perhaps half a day for someone fluent. Most of a real project's time goes to acquiring and understanding the data, and to the feature engineering held back here — not to the modelling.

**Can I use AI to write all of this?**
Yes — it is a delivery block and that is the intent. The skill being built is specifying what you want and verifying the result. Every step in this notebook had a verification attached for exactly that reason. An engineer who cannot tell that a model is silently 80% accurate and catching nothing is not made safe by better code generation.

---

# Session Conclusion

Session 4 covered a great deal, so here is the compression.

**Notebook 01** established that "classification" is five different problems wearing one word. Recognising which one you face determines your metrics before it determines anything else.

**Notebook 02** opened the two boxes you had been calling blind. Logistic regression turns a straight line into a probability, and its coefficients are readable as log-odds — that readability is a business feature, not a consolation prize. Decision trees ask sequential questions, find interactions no line can represent, and memorise without hesitation if you let them.

**Notebook 03** mapped the rest of the territory by *assumption* rather than by derivation, because the assumption is what predicts the failure. It also paid off a promise from Session 3: named `Pipeline` steps make tuning legible, and `best_score_` is optimistic by construction.

**Notebook 04** assembled all of it into a workflow that produces a defensible recommendation rather than a leaderboard.

The thread running through all four is the one from Session 3, unchanged: **a number is not a result until you know what it would look like if you were fooling yourself.** Session 4 added models to that discipline. It did not replace it.

---

# Transition to Session 5

Your bake-off had a conspicuous absence. Every model here was a **single** model — one line, one tree, one vote.

Session 5 asks what happens when you build hundreds of deliberately mediocre trees and let them vote. The answer is counter-intuitive enough to be worth a session, and the result is the family that still wins most tabular problems in 2026 — including, most likely, the Employee Turnover CEP.

You will meet `RandomForestClassifier` properly, then `GradientBoostingClassifier` (which the CEP names explicitly), then XGBoost and LightGBM (which the market names explicitly). And you will settle the logistic-versus-tree question from today with the statistical comparison you built in Session 3, rather than by eyeballing two numbers.

**Next:** `ML_S5_01_*.ipynb`.
